# 🏢 Insurance Fabric Accelerator — Fabric Native Utilities

**This notebook is the foundation for ALL other notebooks in the accelerator.**

## Fabric Built-In Features Used (DO NOT REIMPLEMENT)

| Feature | Fabric Built-In | Notes |
|---------|----------------|-------|
| `spark` session | Pre-initialized | Never import SparkSession |
| Secrets | `notebookutils.credentials.getSecret()` | Never roll your own |
| Tokens | `notebookutils.credentials.getToken()` | Workspace Identity (MSI) |
| File I/O | `notebookutils.fs.*` | OneLake native |
| Orchestration | `notebookutils.notebook.run()` | Built-in DAG |
| Email | `notebookutils.notification.sendMail()` | Built-in |
| Delta maintenance | `OPTIMIZE`, `VACUUM` | V-Order automatic |
| Schema evolution | `.option('mergeSchema', 'true')` | Built-in |
| Time travel | `VERSION AS OF` | Built-in |
| MLflow | `import mlflow` | Pre-installed in Fabric |
| Semantic Link | `import sempy` | Pre-installed in Fabric |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1: Spark Configuration (%%configure equivalent)              ║
# ║  These are Fabric-native Spark settings — applied automatically.   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# V-Order is Fabric's columnar optimization — automatic on write
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")

# Optimize Write — auto-coalesces small files on write (Fabric built-in)
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

# Low Shuffle Merge — Fabric-specific optimization for Delta MERGE
spark.conf.set("spark.microsoft.delta.merge.lowShuffle.enabled", "true")

# Adaptive Query Execution — built-in in Fabric Spark
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Auto schema merge for evolving sources
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print("✅ Fabric Spark configuration applied")
print(f"   Spark version: {spark.version}")
print(f"   V-Order: {spark.conf.get('spark.sql.parquet.vorder.enabled')}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2: Fabric-Native Secret & Token Management                   ║
# ║  Uses notebookutils.credentials — NEVER reimplement.               ║
# ╚══════════════════════════════════════════════════════════════════════╝

def get_secret(key_vault_url: str, secret_name: str) -> str:
    """Retrieve secret from Azure Key Vault using Fabric built-in credentials.
    
    In Fabric, notebookutils.credentials.getSecret() uses the Workspace Identity
    (Managed Identity) — no service principal or API keys needed.
    
    Args:
        key_vault_url: Full Key Vault URL (e.g., 'https://myvault.vault.azure.net/')
        secret_name: Name of the secret in Key Vault
    
    Returns:
        Secret value as string
    """
    return notebookutils.credentials.getSecret(key_vault_url, secret_name)


def get_token(resource: str = "https://api.fabric.microsoft.com") -> str:
    """Get OAuth token using Fabric Workspace Identity (Managed Identity).
    
    Supports any Azure resource endpoint:
      - https://api.fabric.microsoft.com  (Fabric REST API)
      - https://storage.azure.com         (Azure Storage)
      - https://database.windows.net      (Azure SQL)
      - https://cognitiveservices.azure.com (Azure AI)
    """
    return notebookutils.credentials.getToken(resource)


# Metadata schema constant — used by all notebooks
METADATA_SCHEMA = "insurance_metadata"
MONITORING_SCHEMA = "insurance_monitoring"
SECURITY_SCHEMA = "insurance_security"
UNSTRUCTURED_SCHEMA = "insurance_unstructured"

print("✅ Secrets & token helpers loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3: Fabric REST API Client                                     ║
# ║  All Fabric management operations go through this client.           ║
# ║  Auth: Workspace Identity via notebookutils.credentials.getToken()  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import requests
import json
import time
from typing import Optional, Dict, List, Any


class FabricAPI:
    """Thin wrapper around Fabric REST API using Workspace Identity."""
    
    BASE = "https://api.fabric.microsoft.com/v1"
    
    @property
    def _headers(self) -> dict:
        return {
            "Authorization": f"Bearer {get_token()}",
            "Content-Type": "application/json"
        }
    
    def get(self, path: str, **kwargs) -> dict:
        r = requests.get(f"{self.BASE}{path}", headers=self._headers, timeout=60, **kwargs)
        r.raise_for_status()
        return r.json() if r.content else {}
    
    def post(self, path: str, body: dict = None, **kwargs) -> dict:
        r = requests.post(f"{self.BASE}{path}", headers=self._headers,
                          json=body or {}, timeout=120, **kwargs)
        r.raise_for_status()
        return r.json() if r.content else {}
    
    def patch(self, path: str, body: dict = None) -> dict:
        r = requests.patch(f"{self.BASE}{path}", headers=self._headers,
                           json=body or {}, timeout=60)
        r.raise_for_status()
        return r.json() if r.content else {}
    
    def delete(self, path: str) -> None:
        r = requests.delete(f"{self.BASE}{path}", headers=self._headers, timeout=60)
        r.raise_for_status()
    
    def post_lro(self, path: str, body: dict = None,
                 poll_sec: int = 5, max_polls: int = 60) -> dict:
        """POST with long-running operation polling."""
        r = requests.post(f"{self.BASE}{path}", headers=self._headers,
                          json=body or {}, timeout=120)
        if r.status_code == 202:
            op_url = r.headers.get("Location") or r.headers.get("Operation-Location")
            if op_url:
                for _ in range(max_polls):
                    time.sleep(poll_sec)
                    poll = requests.get(op_url, headers=self._headers, timeout=60)
                    result = poll.json()
                    if result.get("status") in ("Succeeded", "succeeded"):
                        return result
                    if result.get("status") in ("Failed", "failed"):
                        raise RuntimeError(f"LRO failed: {result}")
        r.raise_for_status()
        return r.json() if r.content else {}


# Singleton — use `fabric` in all notebooks
fabric = FabricAPI()
print("✅ Fabric REST API client ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 4: OneLake File Operations (notebookutils.fs)                 ║
# ║  Built-in — never use os.path or shutil in Fabric.                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

def onelake_ls(path: str) -> list:
    """List files/folders at an OneLake path."""
    return [f.name for f in notebookutils.fs.ls(path)]


def onelake_cp(src: str, dst: str, recurse: bool = False):
    """Copy files within OneLake."""
    notebookutils.fs.cp(src, dst, recurse)


def onelake_rm(path: str, recurse: bool = False):
    """Remove files from OneLake."""
    notebookutils.fs.rm(path, recurse)


def onelake_mkdirs(path: str):
    """Create directory in OneLake."""
    notebookutils.fs.mkdirs(path)


def onelake_read_text(path: str) -> str:
    """Read text file from OneLake."""
    return notebookutils.fs.head(path, 1000000)


print("✅ OneLake file operations loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 5: Notebook Orchestration (notebookutils.notebook)            ║
# ║  Run child notebooks, parallel execution — all built-in.            ║
# ╚══════════════════════════════════════════════════════════════════════╝

def run_child_notebook(notebook_name: str, params: dict = None,
                       timeout_sec: int = 600) -> str:
    """Run a Fabric notebook and return its exit value.
    
    This uses Fabric's built-in notebook orchestration — NOT subprocess.
    The child notebook runs in the same Spark session by default.
    """
    return notebookutils.notebook.run(
        notebook_name, timeout_sec, params or {}
    )


def run_notebooks_parallel(configs: list) -> list:
    """Run multiple notebooks in parallel using Fabric built-in DAG.
    
    Args:
        configs: List of dicts with keys:
            - name: activity name
            - notebook: notebook path
            - parameters: dict of parameters (optional)
            - timeout: seconds (optional, default 600)
    """
    dag = {
        "activities": [
            {
                "name": c["name"],
                "path": c["notebook"],
                "args": c.get("parameters", {}),
                "timeoutPerCellInSeconds": c.get("timeout", 600),
                "retry": c.get("retry", 0),
            }
            for c in configs
        ],
        "timeoutInSeconds": max(c.get("timeout", 600) for c in configs) * 2,
        "concurrency": len(configs),
    }
    return notebookutils.notebook.runMultiple(dag)


print("✅ Notebook orchestration helpers loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 6: Notifications (notebookutils.notification)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

def send_email(to: str, subject: str, body: str):
    """Send email using Fabric built-in notification service."""
    notebookutils.notification.sendMail(
        to=to, subject=subject, body=body
    )


print("✅ Notification helpers loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 7: Delta Table Maintenance (built-in OPTIMIZE + VACUUM)       ║
# ╚══════════════════════════════════════════════════════════════════════╝

def optimize_table(table: str, z_order_cols: list = None):
    """Run Fabric-native OPTIMIZE with V-Order (applied automatically)."""
    if z_order_cols:
        spark.sql(f"OPTIMIZE {table} ZORDER BY ({','.join(z_order_cols)})")
    else:
        spark.sql(f"OPTIMIZE {table}")
    print(f"  ✅ OPTIMIZE {table}")


def vacuum_table(table: str, hours: int = 168):
    """Run Fabric-native VACUUM to reclaim storage."""
    spark.sql(f"VACUUM {table} RETAIN {hours} HOURS")
    print(f"  ✅ VACUUM {table} (retain {hours}h)")


def table_history(table: str, limit: int = 10):
    """Show Delta table version history (built-in)."""
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT {limit}")


def time_travel(table: str, version: int):
    """Query a specific table version (built-in Delta time travel)."""
    return spark.sql(f"SELECT * FROM {table} VERSION AS OF {version}")


print("✅ Delta maintenance helpers loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 8: OneLake Path Helpers                                       ║
# ╚══════════════════════════════════════════════════════════════════════╝

def tables_path(workspace: str, lakehouse: str) -> str:
    """Get abfss path for lakehouse Tables (Delta tables)."""
    return (f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/"
            f"{lakehouse}.Lakehouse/Tables")


def files_path(workspace: str, lakehouse: str) -> str:
    """Get abfss path for lakehouse Files (unstructured data)."""
    return (f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/"
            f"{lakehouse}.Lakehouse/Files")


def get_config(key: str, env: str = "prod") -> str:
    """Read a config value from the metadata environment_config table."""
    row = spark.sql(f"""
        SELECT config_value FROM {METADATA_SCHEMA}.environment_config
        WHERE config_key = '{key}' AND environment = '{env}'
          AND is_active = TRUE
        LIMIT 1
    """).first()
    return row["config_value"] if row else None


print("✅ Path & config helpers loaded")
print("\n🎉 Fabric Native Utilities notebook ready. Run %run to use in other notebooks.")